# Rest & Travel Analysis (Phase 2, Area 1)

**Goal:** Determine whether rest days, back-to-back status, travel distance, and timezone crossings measurably affect shot quality and game outcomes.

**Key questions:**
1. What does the rest-day distribution look like? (~15-20% back-to-back expected)
2. Does travel distance or timezone crossing matter more?
3. Does rest/travel predict shot quality at all?
4. Are rest and b2b redundant?
5. Does schedule density (3-in-4, 4-in-6) warrant future feature work?

**Data source:** `game_context` and `shot_events` tables in `data/nhl_data.db`

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Add src/ to path — handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH

sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")

## 1. Data availability check

Verify we have `game_context` rows populated.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM game_context")
total_ctx = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM game_context WHERE home_rest_days IS NOT NULL")
with_rest = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM game_context WHERE travel_distance_km IS NOT NULL")
with_travel = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM games")
total_games = cur.fetchone()[0]

print(f"Games in games table:        {total_games:,}")
print(f"Game context rows:           {total_ctx:,}")
print(f"  with rest data:            {with_rest:,}")
print(f"  with travel data:          {with_travel:,}")

if total_ctx == 0:
    print("\n*** No game_context data. Run the scraper to populate games + game_context. ***")

## 2. Rest-day distribution

What does the rest-day distribution look like for home and away teams? Expectation: ~15-20% of games are back-to-backs (rest_days = 1).

In [ ]:
MAX_REST_DISPLAY = 10  # cap histogram at 10 days for readability

cur.execute("""
    SELECT home_rest_days, COUNT(*) AS cnt
    FROM game_context
    WHERE home_rest_days IS NOT NULL AND home_rest_days BETWEEN 1 AND ?
    GROUP BY home_rest_days
    ORDER BY home_rest_days
""", (MAX_REST_DISPLAY,))
home_rows = cur.fetchall()

cur.execute("""
    SELECT away_rest_days, COUNT(*) AS cnt
    FROM game_context
    WHERE away_rest_days IS NOT NULL AND away_rest_days BETWEEN 1 AND ?
    GROUP BY away_rest_days
    ORDER BY away_rest_days
""", (MAX_REST_DISPLAY,))
away_rows = cur.fetchall()

if home_rows and away_rows:
    home_days = [r[0] for r in home_rows]
    home_counts = [r[1] for r in home_rows]
    away_days = [r[0] for r in away_rows]
    away_counts = [r[1] for r in away_rows]

    home_total = sum(home_counts)
    away_total = sum(away_counts)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(home_days, home_counts, color="#3498db", alpha=0.8)
    ax1.set_xlabel("Rest days (home team)")
    ax1.set_ylabel("Number of games")
    ax1.set_title("Home Team Rest-Day Distribution")
    ax1.set_xticks(range(1, MAX_REST_DISPLAY + 1))
    for d, c in zip(home_days, home_counts):
        ax1.text(d, c + home_total * 0.005, f"{c/home_total:.1%}", ha="center", fontsize=8)

    ax2.bar(away_days, away_counts, color="#e74c3c", alpha=0.8)
    ax2.set_xlabel("Rest days (away team)")
    ax2.set_ylabel("Number of games")
    ax2.set_title("Away Team Rest-Day Distribution")
    ax2.set_xticks(range(1, MAX_REST_DISPLAY + 1))
    for d, c in zip(away_days, away_counts):
        ax2.text(d, c + away_total * 0.005, f"{c/away_total:.1%}", ha="center", fontsize=8)

    fig.tight_layout()
    plt.show()

    home_b2b = home_counts[0] if home_days[0] == 1 else 0
    away_b2b = away_counts[0] if away_days[0] == 1 else 0
    print(f"Home back-to-back rate: {home_b2b}/{home_total} = {home_b2b/home_total:.1%}")
    print(f"Away back-to-back rate: {away_b2b}/{away_total} = {away_b2b/away_total:.1%}")
else:
    print("No rest data available.")

## 3. Rest advantage and goal rate

Does the team with more rest win more often? Compare goal rate by rest advantage (home_rest - away_rest).

In [ ]:
cur.execute("""
    SELECT
        gc.rest_advantage,
        COUNT(DISTINCT gc.game_id) AS games,
        COUNT(se.shot_event_id) AS total_shots,
        SUM(se.is_goal) AS total_goals,
        ROUND(CAST(SUM(se.is_goal) AS REAL) / COUNT(se.shot_event_id), 4) AS goal_rate,
        ROUND(AVG(se.distance_to_goal), 1) AS avg_distance
    FROM game_context gc
    JOIN shot_events se ON gc.game_id = se.game_id
    WHERE gc.rest_advantage IS NOT NULL
      AND gc.rest_advantage BETWEEN -5 AND 5
    GROUP BY gc.rest_advantage
    ORDER BY gc.rest_advantage
""")

rows = cur.fetchall()
if rows:
    adv = [r[0] for r in rows]
    games = [r[1] for r in rows]
    goal_rates = [r[4] for r in rows]
    avg_dists = [r[5] for r in rows]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    colors = ["#e74c3c" if a < 0 else "#3498db" if a > 0 else "#95a5a6" for a in adv]
    ax1.bar(adv, goal_rates, color=colors, alpha=0.8)
    ax1.set_xlabel("Rest advantage (home - away days)")
    ax1.set_ylabel("Goal rate (all shots)")
    ax1.set_title("Goal Rate by Rest Advantage")
    ax1.axvline(x=0, color="black", linewidth=0.5, linestyle="--")

    ax2.bar(adv, games, color="#2ecc71", alpha=0.8)
    ax2.set_xlabel("Rest advantage (home - away days)")
    ax2.set_ylabel("Number of games")
    ax2.set_title("Sample Size by Rest Advantage")

    fig.tight_layout()
    plt.show()

    print(f"{'Rest Adv':>9} {'Games':>8} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10}")
    print("-" * 60)
    for r in rows:
        print(f"{r[0]:>9} {r[1]:>8,} {r[2]:>10,} {r[3]:>8,} {r[4]:>10.4f} {r[5]:>10.1f}")
else:
    print("No rest advantage data available.")

## 4. Back-to-back effect on shot quality

Compare shot distance and goal rate for back-to-back vs. rested games, split by home/away.

In [ ]:
cur.execute("""
    SELECT
        CASE WHEN gc.home_is_back_to_back = 1 THEN 'Home B2B' ELSE 'Home Rested' END AS label,
        COUNT(se.shot_event_id) AS shots,
        SUM(se.is_goal) AS goals,
        ROUND(CAST(SUM(se.is_goal) AS REAL) / COUNT(se.shot_event_id), 4) AS goal_rate,
        ROUND(AVG(se.distance_to_goal), 1) AS avg_dist,
        COUNT(DISTINCT gc.game_id) AS games
    FROM game_context gc
    JOIN shot_events se ON gc.game_id = se.game_id
    JOIN games g ON gc.game_id = g.game_id
    WHERE gc.home_is_back_to_back IS NOT NULL
      AND se.shooting_team_id = g.home_team_id
    GROUP BY gc.home_is_back_to_back
""")
home_rows = cur.fetchall()

cur.execute("""
    SELECT
        CASE WHEN gc.away_is_back_to_back = 1 THEN 'Away B2B' ELSE 'Away Rested' END AS label,
        COUNT(se.shot_event_id) AS shots,
        SUM(se.is_goal) AS goals,
        ROUND(CAST(SUM(se.is_goal) AS REAL) / COUNT(se.shot_event_id), 4) AS goal_rate,
        ROUND(AVG(se.distance_to_goal), 1) AS avg_dist,
        COUNT(DISTINCT gc.game_id) AS games
    FROM game_context gc
    JOIN shot_events se ON gc.game_id = se.game_id
    JOIN games g ON gc.game_id = g.game_id
    WHERE gc.away_is_back_to_back IS NOT NULL
      AND se.shooting_team_id = g.away_team_id
    GROUP BY gc.away_is_back_to_back
""")
away_rows = cur.fetchall()

all_rows = list(home_rows) + list(away_rows)
if all_rows:
    print(f"{'Condition':<16} {'Games':>8} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10}")
    print("-" * 66)
    for r in all_rows:
        print(f"{r[0]:<16} {r[5]:>8,} {r[1]:>10,} {r[2]:>8,} {r[3]:>10.4f} {r[4]:>10.1f}")

    labels = [r[0] for r in all_rows]
    rates = [r[3] for r in all_rows]
    colors = ["#e74c3c" if "B2B" in l else "#3498db" for l in labels]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, rates, color=colors, alpha=0.8)
    ax.set_ylabel("Goal rate")
    ax.set_title("Goal Rate: Back-to-Back vs Rested (by Home/Away)")
    fig.tight_layout()
    plt.show()
else:
    print("No b2b data available.")

## 5. Travel distance distribution and effect

How far do away teams travel? Does travel distance correlate with shot quality?

In [ ]:
cur.execute("""
    SELECT travel_distance_km FROM game_context
    WHERE travel_distance_km IS NOT NULL AND travel_distance_km > 0
""")
distances = [r[0] for r in cur.fetchall()]

if distances:
    dist_arr = np.array(distances)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.hist(dist_arr, bins=50, color="#3498db", alpha=0.8, edgecolor="white")
    ax1.set_xlabel("Travel distance (km)")
    ax1.set_ylabel("Number of games")
    ax1.set_title("Travel Distance Distribution")
    ax1.axvline(x=np.median(dist_arr), color="#e74c3c", linestyle="--",
                label=f"Median: {np.median(dist_arr):.0f} km")
    ax1.legend()

    TRAVEL_BUCKETS = [0, 500, 1000, 2000, 3000, 5000]
    bucket_labels = []
    bucket_rates = []
    bucket_games = []

    for i in range(len(TRAVEL_BUCKETS) - 1):
        lo, hi = TRAVEL_BUCKETS[i], TRAVEL_BUCKETS[i + 1]
        cur.execute("""
            SELECT COUNT(se.shot_event_id) AS shots,
                   SUM(se.is_goal) AS goals,
                   COUNT(DISTINCT gc.game_id) AS games
            FROM game_context gc
            JOIN shot_events se ON gc.game_id = se.game_id
            WHERE gc.travel_distance_km > ? AND gc.travel_distance_km <= ?
        """, (lo, hi))
        r = cur.fetchone()
        if r and r[0] > 0:
            bucket_labels.append(f"{lo}-{hi}")
            bucket_rates.append(r[1] / r[0])
            bucket_games.append(r[2])

    ax2.bar(bucket_labels, bucket_rates, color="#2ecc71", alpha=0.8)
    ax2.set_xlabel("Travel distance bucket (km)")
    ax2.set_ylabel("Goal rate")
    ax2.set_title("Goal Rate by Travel Distance")
    for i, (lbl, g) in enumerate(zip(bucket_labels, bucket_games)):
        ax2.text(i, bucket_rates[i] + 0.001, f"n={g:,}", ha="center", fontsize=8)

    fig.tight_layout()
    plt.show()

    print(f"Travel distance stats: median={np.median(dist_arr):.0f} km, "
          f"mean={np.mean(dist_arr):.0f} km, max={np.max(dist_arr):.0f} km")
else:
    print("No travel distance data available.")

## 6. Timezone delta vs. travel distance

Are timezone crossing and travel distance collinear, or do they capture different signals? If highly correlated, one may be redundant.

In [ ]:
cur.execute("""
    SELECT travel_distance_km, timezone_delta
    FROM game_context
    WHERE travel_distance_km IS NOT NULL
      AND timezone_delta IS NOT NULL
      AND travel_distance_km > 0
""")
rows = cur.fetchall()

if rows:
    travel = np.array([r[0] for r in rows])
    tz = np.array([r[1] for r in rows])

    correlation = np.corrcoef(travel, np.abs(tz))[0, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.scatter(travel, tz, alpha=0.15, s=5, color="#3498db")
    ax1.set_xlabel("Travel distance (km)")
    ax1.set_ylabel("Timezone delta (hours, home - away)")
    ax1.set_title(f"Travel Distance vs Timezone Delta (r={correlation:.3f} with |tz|)")

    cur.execute("""
        SELECT gc.timezone_delta,
               COUNT(se.shot_event_id) AS shots,
               SUM(se.is_goal) AS goals,
               COUNT(DISTINCT gc.game_id) AS games
        FROM game_context gc
        JOIN shot_events se ON gc.game_id = se.game_id
        WHERE gc.timezone_delta IS NOT NULL
        GROUP BY gc.timezone_delta
        ORDER BY gc.timezone_delta
    """)
    tz_rows = cur.fetchall()
    if tz_rows:
        tz_vals = [r[0] for r in tz_rows]
        tz_rates = [r[2] / r[1] if r[1] > 0 else 0 for r in tz_rows]
        tz_games = [r[3] for r in tz_rows]

        colors = ["#e74c3c" if t < 0 else "#3498db" if t > 0 else "#95a5a6" for t in tz_vals]
        ax2.bar(tz_vals, tz_rates, color=colors, alpha=0.8)
        ax2.set_xlabel("Timezone delta (hours)")
        ax2.set_ylabel("Goal rate")
        ax2.set_title("Goal Rate by Timezone Delta")
        for i, (tv, g) in enumerate(zip(tz_vals, tz_games)):
            ax2.text(tv, tz_rates[i] + 0.001, f"n={g:,}", ha="center", fontsize=7, rotation=45)

    fig.tight_layout()
    plt.show()

    print(f"Correlation between travel distance and |timezone delta|: {correlation:.3f}")
    print(f"  If > 0.8: likely redundant, pick one")
    print(f"  If < 0.5: independent signals, keep both")
else:
    print("No travel/timezone data available.")

## 7. Feature correlation matrix

Check pairwise correlations among all rest/travel features to identify redundancy.

In [ ]:
cur.execute("""
    SELECT home_rest_days, away_rest_days, rest_advantage,
           home_is_back_to_back, away_is_back_to_back,
           travel_distance_km, timezone_delta
    FROM game_context
    WHERE home_rest_days IS NOT NULL
      AND away_rest_days IS NOT NULL
      AND travel_distance_km IS NOT NULL
      AND timezone_delta IS NOT NULL
""")
rows = cur.fetchall()

if rows:
    col_names = ["home_rest", "away_rest", "rest_adv",
                 "home_b2b", "away_b2b", "travel_km", "tz_delta"]
    data = np.array(rows, dtype=float)

    corr = np.corrcoef(data.T)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(col_names)))
    ax.set_xticklabels(col_names, rotation=45, ha="right")
    ax.set_yticks(range(len(col_names)))
    ax.set_yticklabels(col_names)
    ax.set_title("Rest/Travel Feature Correlation Matrix")

    for i in range(len(col_names)):
        for j in range(len(col_names)):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center",
                    fontsize=9, color="white" if abs(corr[i, j]) > 0.5 else "black")

    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()

    print("Look for |r| > 0.7 pairs — candidates for dropping one feature.")
    print("Expected: home_rest vs home_b2b should be highly correlated (b2b = rest==1).")
else:
    print("No complete game_context data available.")

## 8. Summary and recommendations

**Interpret the results above to answer:**

1. **Rest-day distribution:** Does ~15-20% b2b frequency hold? If so, there's enough data for b2b to be a useful feature.

2. **Rest advantage signal:** Is there a monotonic trend in goal rate by rest advantage? If the effect is <0.5% goal rate difference, it may not improve an xG model.

3. **Travel vs. timezone:** If correlation > 0.8, drop one (prefer timezone delta — it's more interpretable). If < 0.5, keep both.

4. **Feature subset for xG:** Based on the correlation matrix, recommend a non-redundant subset. Likely candidates: `rest_advantage` (or just `home_b2b + away_b2b`), `travel_distance_km`, `timezone_delta`.

5. **Schedule density (deferred):** If b2b shows signal, that motivates the deferred 3-in-4 / 4-in-6 work. See memory note for details.

In [ ]:
conn.close()
print("Done.")